In [7]:
import pandas as pd 
import numpy as np  
from scipy.stats import chisquare, ttest_ind
from statsmodels.stats.power import TTestIndPower 
from pathlib import Path
import os 
import matplotlib.pyplot as plt 

In [8]:
PROJECT_ROOT = Path.cwd().resolve()

print("Project root:", PROJECT_ROOT)

Project root: /Users/scottbelarmino/ds_decision_science_exp_repo/free_shipping_threshold_experiment


In [9]:
# load experiment/synthetic dataset 
 
DATA_DIR = PROJECT_ROOT / "data"
df = pd.read_csv(DATA_DIR / "synthetic/free_shipping_experiment_sessions.csv")

df.head()

,session_id,visitor_id,session_date,period,region,test_arm,conversion_multiplier,basket_multiplier,threshold_arm,free_shipping_threshold,...,order_id,units,price_per_unit,product_revenue,shipping_revenue,total_revenue,cogs,shipping_cost,contribution_margin,qualified_for_free_shipping
0,1,650921,2025-01-01,pre,West,t65,1.306001,1.076006,pre,50.0,...,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,2,373732,2025-01-01,pre,West,t65,0.988317,1.140570,pre,50.0,...,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,3,338105,2025-01-01,pre,Central,t35,1.227563,0.882506,pre,50.0,...,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,4,156611,2025-01-01,pre,West,t35,1.123940,1.324733,pre,50.0,...,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,5,624327,2025-01-01,pre,West,t50,0.790399,0.764117,pre,50.0,...,NaN,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [10]:
# isolating the test period into test_df

test_df = df[df['period'] == 'test'].copy()  
test_df.shape

(449542, 23)

# Power Analysis  

---

Purpose:  
    As part of the experiment design phase, we need to determine the experiment will collect enough data to detect meaningful treatment effect. The power analysis will answer:   
    
        1. How much data do we need?  
        2. How long do we need to run the experiment?  

The power analysis will ensure the experiment has a good chance of detecting meaningful improvement in contribution margin per session (primary metric)

In [22]:

# baseline arm
baseline = test_df.loc[test_df["threshold_arm"] == "t50", "contribution_margin"]

baseline_mean = baseline.mean()
baseline_std = baseline.std(ddof=1)

# minimum detectable effect (mde) - difference between minimum expected improvement and baseline
mde = 0.0245

# standardized effect size
effect_size = mde / baseline_std

analysis = TTestIndPower()

sample_size_per_arm = analysis.solve_power(
    effect_size=effect_size,
    power=0.80,
    alpha=0.05,
    ratio=1.0,
    alternative="two-sided"
)

print("Baseline mean CM/session:", round(baseline_mean, 4))
print("Baseline std dev:", round(baseline_std, 4))
print("MDE:", mde)
print("Percent improvement:", round(mde / baseline_mean * 100, 2), "%")
print("Required sample size per arm:", round(sample_size_per_arm))

Baseline mean CM/session: 0.3316
Baseline std dev: 2.3975
MDE: 0.0245
Percent improvement: 7.39 %
Required sample size per arm: 150321


In [23]:
avg_sessions_per_day = 10000
n_arms = 3

total_required_sessions = sample_size_per_arm * n_arms
days_required = total_required_sessions / avg_sessions_per_day

print("Total required sessions:", round(total_required_sessions))
print("Estimated runtime (days):", round(days_required, 1))

Total required sessions: 450962
Estimated runtime (days): 45.1


## Business Impact  

---

With the assumption of approx. 10K sessions per day, 3.65M sessions per year:  

If MDE ~ 0.025, the minimum annual contribution margin impact would be:  

        $0.025 x 3,650,000 = $91,250

This translates to: `Can the experiment detect a change worth $91,000 per year or more?`


